In [115]:
import yfinance as yf
import numpy as np
# import pandas_ta as ta
import pandas_ta_classic as ta
import pandas as pd

In [116]:
TICKER = "EURUSD=X"

# df = yf.download(TICKER, period="max", interval="5m")

# df.columns.names = [None, None]

# df.columns = df.columns.get_level_values(0)

# df = df.drop(columns=["Volume"])

df = pd.read_csv("eurusd-m5-bid-2020-01-01-2026-08-15.csv")

df.set_index("Datetime", inplace=True)

df = df[["Open", "High", "Low", "Close"]]

df.index = pd.to_datetime(df.index, unit="ms")

# df = df.tail(10000) # for testing

df

,Open,High,Low,Close
Datetime,,,,
2020-01-01 00:00:00,1.12139,1.12139,1.12139,1.12139
2020-01-01 00:05:00,1.12139,1.12139,1.12139,1.12139
2020-01-01 00:10:00,1.12139,1.12139,1.12139,1.12139
2020-01-01 00:15:00,1.12139,1.12139,1.12139,1.12139
2020-01-01 00:20:00,1.12139,1.12139,1.12139,1.12139
...,...,...,...,...
2026-08-14 20:35:00,1.15684,1.15688,1.15681,1.15682
2026-08-14 20:40:00,1.15683,1.15686,1.15672,1.15674
2026-08-14 20:45:00,1.15674,1.15686,1.15669,1.15675


In [117]:
GRID_GAP = 0.001
ATR_LENGTH = 16

In [118]:
las_visited_grid_level = df["Close"].iloc[0]

Signal = np.zeros(len(df), dtype=int)

Signal[0] = 1

for i, _close in enumerate(df["Close"]):
    if abs(_close - las_visited_grid_level) > GRID_GAP:
        Signal[i] = 1
        las_visited_grid_level = _close

df["Signal"] = Signal

df[df["Signal"] == 1]

,Open,High,Low,Close,Signal
Datetime,,,,,
2020-01-01 00:00:00,1.12139,1.12139,1.12139,1.12139,1
2020-01-02 02:45:00,1.12239,1.12244,1.12238,1.12243,1
2020-01-02 03:40:00,1.12160,1.12161,1.12131,1.12135,1
2020-01-02 04:55:00,1.12046,1.12046,1.12008,1.12012,1
2020-01-02 08:00:00,1.12099,1.12137,1.12087,1.12135,1
...,...,...,...,...,...
2026-08-14 06:55:00,1.15420,1.15444,1.15413,1.15443,1
2026-08-14 08:05:00,1.15526,1.15553,1.15524,1.15551,1
2026-08-14 11:30:00,1.15627,1.15677,1.15620,1.15673,1


In [119]:
dfpl = df[:].copy()

def SIGNAL(): return dfpl.Signal

dfpl['ATR'] = ta.atr(high = dfpl.High, low = dfpl.Low, close = dfpl.Close, length = ATR_LENGTH)

dfpl.dropna(inplace=True)

In [120]:
from backtesting import Strategy, Backtest

MAX_OPEN_TRADES = 10000

class MyStrat(Strategy):
    volume = 1
    rrr = 0.5

    def init(self):
        super().init()
        self.signal = self.I(SIGNAL)

    def transact(self):
        sl_distance = 1.5 * GRID_GAP
        # sl_distance = 20 * self.data.ATR[-1]

        self.sell(
            sl=self.data.Close[-1] + sl_distance,
            tp=self.data.Close[-1] - sl_distance * self.rrr,
            size=self.volume
        )

        self.buy(
            sl=self.data.Close[-1] - sl_distance,
            tp=self.data.Close[-1] + sl_distance * self.rrr,
            size=self.volume
        )

    def next(self):
        super().next()

        if self.signal==1 and len(self.trades) <= MAX_OPEN_TRADES: self.transact()

bt = Backtest(dfpl, MyStrat, cash=100, margin=1/100, hedging=True, exclusive_orders=False, finalize_trades=True)

stat = bt.run()

stat

Start                     2020-01-01 01:15:00
End                       2026-08-14 20:55:00
Duration                   2417 days 19:40:00
Exposure Time [%]                    87.89926
Equity Final [$]                    100.14163
Equity Peak [$]                      100.1441
Return [%]                            0.14163
Buy & Hold Return [%]                 3.15947
Return (Ann.) [%]                     0.01725
Volatility (Ann.) [%]                 0.05649
CAGR [%]                              0.02138
Sharpe Ratio                           0.3053
Sortino Ratio                          0.4121
Calmar Ratio                          0.13512
Alpha [%]                             0.14155
Beta                                  0.00003
Max. Drawdown [%]                    -0.12765
Avg. Drawdown [%]                    -0.00168
Max. Drawdown Duration     1381 days 04:15:00
Avg. Drawdown Duration        5 days 08:31:00
# Trades                                51624
Win Rate [%]                      

In [121]:
import backtesting

backtesting.set_bokeh_output(notebook=False)

bt.plot(show_legend=False, plot_width=None, plot_equity=True, plot_return=False, 
        
plot_pl=False, plot_volume=False, plot_drawdown=False, smooth_equity=False, relative_equity=True, 

superimpose=True, resample=False, reverse_indicators=False, open_browser=True)

GridPlot(id='p3271', ...)

gio: file:///home/mod7ex/projects/algo-strategies/Grid/MyStrat.html: Failed to find default application for content type ‘text/html’


In [122]:
stat._trades.sort_values(by="EntryBar").head(20)

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,Entry_SIGNAL,Exit_SIGNAL
1,1,307,329,1.12242,1.12093,1.12093,1.12318,-0.00149,0.0,-0.001327,2020-01-02 02:50:00,2020-01-02 04:40:00,0 days 01:50:00,None,0,0
0,-1,307,315,1.12242,1.12168,1.12393,1.12168,0.00074,0.0,0.000659,2020-01-02 02:50:00,2020-01-02 03:30:00,0 days 00:40:00,None,0,0
2,-1,318,331,1.12134,1.12060,1.12285,1.12060,0.00074,0.0,0.000660,2020-01-02 03:45:00,2020-01-02 04:50:00,0 days 01:05:00,None,0,0
6,1,318,396,1.12134,1.11985,1.11985,1.12210,-0.00149,0.0,-0.001329,2020-01-02 03:45:00,2020-01-02 10:15:00,0 days 06:30:00,None,0,0
7,-1,333,398,1.12011,1.11937,1.12162,1.11937,0.00074,0.0,0.000661,2020-01-02 05:00:00,2020-01-02 10:25:00,0 days 05:25:00,None,0,0
3,1,333,357,1.12011,1.12087,1.11862,1.12087,0.00076,0.0,0.000679,2020-01-02 05:00:00,2020-01-02 07:00:00,0 days 02:00:00,None,0,0
5,1,370,396,1.12134,1.11985,1.11985,1.12210,-0.00149,0.0,-0.001329,2020-01-02 08:05:00,2020-01-02 10:15:00,0 days 02:10:00,None,0,0
4,-1,370,370,1.12134,1.12060,1.12285,1.12060,0.00074,0.0,0.000660,2020-01-02 08:05:00,2020-01-02 08:05:00,0 days 00:00:00,None,0,0
8,-1,391,398,1.12031,1.11957,1.12182,1.11957,0.00074,0.0,0.000661,2020-01-02 09:50:00,2020-01-02 10:25:00,0 days 00:35:00,None,0,0
9,1,391,411,1.12031,1.11882,1.11882,1.12107,-0.00149,0.0,-0.001330,2020-01-02 09:50:00,2020-01-02 11:30:00,0 days 01:40:00,None,0,1
